In [78]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models, layers, callbacks
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from collections import Counter

In [79]:
df=pd.read_csv("cleaned_dataset_taiwan_2months.csv")
df.head()

,date,sitename,county,aqi,status,so2,co,o3,pm10,pm2.5,no2,nox,no,siteid
0,2024-08-31 23:00:00,Hukou,Hsinchu County,62.0,Moderate,0.9,0.17,35.0,18.0,17.0,2.3,2.6,0.3,22
1,2024-08-31 23:00:00,Zhongming,Taichung City,50.0,Good,1.6,0.32,27.9,27.0,14.0,7.6,9.3,1.6,31
2,2024-08-31 23:00:00,Zhudong,Hsinchu County,45.0,Good,0.4,0.17,25.1,21.0,13.0,2.9,4.1,1.1,23
3,2024-08-31 23:00:00,Hsinchu,Hsinchu City,42.0,Good,0.8,0.20,30.0,19.0,10.0,4.0,4.8,0.7,24
4,2024-08-31 23:00:00,Toufen,Miaoli County,50.0,Good,1.0,0.16,33.5,18.0,14.0,1.8,3.1,1.2,25


In [80]:
features = ['aqi', 'so2', 'co', 'o3', 'pm10', 'pm2.5', 'no2', 'nox', 'no']
X = df[features]
y = df['status']

In [81]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [82]:
print(f"Original class distribution: {Counter(y_train)}")

Original class distribution: Counter({np.int64(0): 91885, np.int64(1): 8735, np.int64(2): 111})


In [83]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [84]:
print(f"Resampled class distribution: {Counter(y_train_res)}")

Resampled class distribution: Counter({np.int64(1): 91885, np.int64(0): 91885, np.int64(2): 91885})


In [85]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

In [86]:
# This explicitly defines the input so feature extraction works correctly
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dropout(0.2)(x)
# This layer will be our feature source for the SVM
feature_layer = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(len(le.classes_), activation='softmax')(feature_layer)

In [87]:
model = tf.keras.Sequential([
    layers.Input(shape=(X_train_res.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2), # Prevents overfitting
    
    layers.Dense(1,activation='relu') 
])

In [88]:
model = models.Model(inputs=inputs, outputs=outputs)

In [89]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='loss', 
    factor=0.2, 
    patience=3, 
    min_lr=1e-6
)

my_callbacks = [
    callbacks.EarlyStopping(monitor='loss', patience=7, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath='best_model.keras', monitor='accuracy', save_best_only=True),
    reduce_lr
]

In [90]:
model.compile(optimizer=Adam(learning_rate=0.0005), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_scaled, y_train_res, epochs=30, batch_size=128,validation_split=0.2,callbacks=my_callbacks)

Epoch 1/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9383 - loss: 0.1705 - val_accuracy: 0.9989 - val_loss: 0.0177 - learning_rate: 5.0000e-04
Epoch 2/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9885 - loss: 0.0362 - val_accuracy: 1.0000 - val_loss: 0.0070 - learning_rate: 5.0000e-04
Epoch 3/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9940 - loss: 0.0194 - val_accuracy: 1.0000 - val_loss: 0.0065 - learning_rate: 5.0000e-04
Epoch 4/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9961 - loss: 0.0125 - val_accuracy: 1.0000 - val_loss: 0.0033 - learning_rate: 5.0000e-04
Epoch 5/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9974 - loss: 0.0091 - val_accuracy: 1.0000 - val_loss: 0.0031 - learning_rate: 5.0000e-04
Epoch 6/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9983 - loss: 0.0064 - val_accuracy: 1.0000 - val_loss: 0.0021 - learning_rate: 5.0000e-04
Epoch 7/30
1723/1723 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/ste

In [91]:
# Extract features from the trained neural network (using the second-to-last layer)
feature_extractor = models.Model(inputs=model.input, outputs=model.layers[-2].output)
X_train_features = feature_extractor.predict(X_train_scaled)
X_test_features = feature_extractor.predict(X_test_scaled)

# Train SVM on the extracted features
svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_train_features, y_train_res)

# Evaluate SVM
y_pred_svm = svm_model.predict(X_test_features)
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

8615/8615 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
787/787 ━━━━━━━━━━━━━━━━━━━━ 1s 866us/step
                                precision    recall  f1-score   support

                          Good       1.00      1.00      1.00     22982
                      Moderate       1.00      1.00      1.00      2173
Unhealthy for Sensitive Groups       0.82      1.00      0.90        28

                      accuracy                           1.00     25183
                     macro avg       0.94      1.00      0.97     25183
                  weighted avg       1.00      1.00      1.00     25183



In [92]:
from sklearn.metrics import accuracy_score, precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# Compute global metrics for the SVM model
global_accuracy = accuracy_score(y_test, y_pred_svm)
global_precision = precision_score(y_test, y_pred_svm, average='weighted')
global_f1 = f1_score(y_test, y_pred_svm, average='weighted')

print(f"Global Accuracy: {global_accuracy}")
print(f"Global Precision: {global_precision}")
print(f"Global F1-Score: {global_f1}")

# Add a callback for future model training (e.g., EarlyStopping)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

Global Accuracy: 0.9997617440336735
Global Precision: 0.9998037892042018
Global F1-Score: 0.9997731078551088


In [93]:
import joblib

# Save the SVM model to a file
joblib.dump(svm_model, 'svm_model.pkl')

['svm_model.pkl']

In [94]:
joblib.dump(scaler,'scaler.pkl')
joblib.dump(feature_extractor,'feature_extractor.h5')

['feature_extractor.h5']

In [95]:
print(list(enumerate(le.classes_)))

[(0, 'Good'), (1, 'Moderate'), (2, 'Unhealthy for Sensitive Groups')]
